# Imbalanced Learning: Stroke Prediction

This notebook explores the Stroke Prediction Dataset.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')

from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.metrics import (accuracy_score, precision_score, recall_score,
                             f1_score, roc_auc_score, roc_curve,
                             precision_recall_curve, classification_report,
                             confusion_matrix, average_precision_score)
from imblearn.over_sampling import SMOTE

## 1. Dataset Preparation

### 1a/b. Load the dataset

In [ ]:
df = pd.read_csv('healthcare-dataset-stroke-data.csv')
print(f'Dataset shape: {df.shape}')

### 1c. Explore the data

In [ ]:
df.head()

In [ ]:
df.describe()

In [ ]:
print('Feature types:')
print(df.dtypes)
print()
print('Missing values:')
print(df.isnull().sum())
print()
print('Non-null unique values per column:')
for col in df.columns:
    print(f'  {col}: {df[col].nunique()} unique')

In [ ]:
# Check for 'N/A' strings in bmi
print('BMI N/A count:', (df['bmi'] == 'N/A').sum() if df['bmi'].dtype == object else 0)
print()
print('Class distribution:')
print(df['stroke'].value_counts())
print()
print(f'Stroke rate: {df["stroke"].mean()*100:.2f}%')

### 1d. Preprocess the data

In [ ]:
# Drop 'id' column
df = df.drop('id', axis=1)

# Handle BMI missing values (stored as 'N/A' strings)
df['bmi'] = pd.to_numeric(df['bmi'], errors='coerce')
df['bmi'] = df['bmi'].fillna(df['bmi'].median())

# Remove rows where gender is 'Other' (very few)
df = df[df['gender'] != 'Other']

# Encode categorical variables
cat_cols = ['gender', 'ever_married', 'work_type', 'Residence_type', 'smoking_status']
df_encoded = pd.get_dummies(df, columns=cat_cols, drop_first=True)

print(f'Processed dataset shape: {df_encoded.shape}')
df_encoded.head()

In [ ]:
# Separate features and target
X = df_encoded.drop('stroke', axis=1)
y = df_encoded['stroke']

# Scale numeric features
numeric_cols = ['age', 'avg_glucose_level', 'bmi']
scaler = StandardScaler()
X[numeric_cols] = scaler.fit_transform(X[numeric_cols])

print(f'Features shape: {X.shape}')
print(f'Target distribution:\n{y.value_counts()}')

## 2. Baseline Model Training

### 2a. Train/test split with stratification

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print(f'Training set: {X_train.shape[0]} samples')
print(f'Test set: {X_test.shape[0]} samples')
print(f'Train stroke rate: {y_train.mean()*100:.2f}%')
print(f'Test stroke rate: {y_test.mean()*100:.2f}%')

### 2b/c. Train baseline logistic regression and evaluate

In [ ]:
def evaluate_model(model, X_test, y_test, model_name='Model'):
    """Evaluate a model and return metrics as a dict."""
    y_pred = model.predict(X_test)
    y_prob = model.predict_proba(X_test)[:, 1]
    
    acc = accuracy_score(y_test, y_pred)
    prec = precision_score(y_test, y_pred, zero_division=0)
    rec = recall_score(y_test, y_pred)
    f1 = f1_score(y_test, y_pred)
    auc = roc_auc_score(y_test, y_prob)
    
    print(f'--- {model_name} ---')
    print(f'Accuracy:  {acc:.4f}')
    print(f'Precision: {prec:.4f}')
    print(f'Recall:    {rec:.4f}')
    print(f'F1 Score:  {f1:.4f}')
    print(f'AUC:       {auc:.4f}')
    print()
    
    return {'name': model_name, 'accuracy': acc, 'precision': prec,
            'recall': rec, 'f1': f1, 'auc': auc, 'y_prob': y_prob, 'y_pred': y_pred}

In [ ]:
# Train baseline logistic regression
baseline_model = LogisticRegression(max_iter=1000, random_state=42)
baseline_model.fit(X_train, y_train)

baseline_results = evaluate_model(baseline_model, X_test, y_test, 'Baseline Logistic Regression')

### 2d. ROC and Precision-Recall curves

In [ ]:
def plot_curves(results_list, title_prefix=''):
    """Plot ROC and PR curves for a list of model results."""
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    
    # ROC curve
    for res in results_list:
        fpr, tpr, _ = roc_curve(y_test, res['y_prob'])
        axes[0].plot(fpr, tpr, label=f"{res['name']} (AUC={res['auc']:.3f})")
    axes[0].plot([0, 1], [0, 1], 'k--', alpha=0.5)
    axes[0].set_xlabel('False Positive Rate')
    axes[0].set_ylabel('True Positive Rate')
    axes[0].set_title(f'{title_prefix}ROC Curve')
    axes[0].legend(loc='lower right')
    axes[0].grid(True, alpha=0.3)
    
    # Precision-Recall curve
    for res in results_list:
        prec_vals, rec_vals, _ = precision_recall_curve(y_test, res['y_prob'])
        ap = average_precision_score(y_test, res['y_prob'])
        axes[1].plot(rec_vals, prec_vals, label=f"{res['name']} (AP={ap:.3f})")
    axes[1].set_xlabel('Recall')
    axes[1].set_ylabel('Precision')
    axes[1].set_title(f'{title_prefix}Precision-Recall Curve')
    axes[1].legend(loc='upper right')
    axes[1].grid(True, alpha=0.3)
    
    plt.tight_layout()
    plt.show()

In [ ]:
plot_curves([baseline_results], 'Baseline: ')

**Discussion:** With heavy class imbalance (<5% positive), accuracy is misleadingly high since predicting all negatives yields ~95% accuracy. Precision and recall for the minority class are much more informative. The precision-recall curve is more reliable than ROC in imbalanced settings because ROC can appear optimistic when the number of true negatives is very large.

## 3. Oversampling with SMOTE

### 3a/b. Apply SMOTE and visualize class distribution

In [ ]:
# Apply SMOTE to training set only
smote = SMOTE(random_state=42)
X_train_smote, y_train_smote = smote.fit_resample(X_train, y_train)

# Visualize class distribution before and after
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

axes[0].bar(['No Stroke (0)', 'Stroke (1)'],
            [sum(y_train == 0), sum(y_train == 1)],
            color=['steelblue', 'coral'])
axes[0].set_title('Before SMOTE')
axes[0].set_ylabel('Count')
for i, v in enumerate([sum(y_train == 0), sum(y_train == 1)]):
    axes[0].text(i, v + 20, str(v), ha='center', fontweight='bold')

axes[1].bar(['No Stroke (0)', 'Stroke (1)'],
            [sum(y_train_smote == 0), sum(y_train_smote == 1)],
            color=['steelblue', 'coral'])
axes[1].set_title('After SMOTE')
axes[1].set_ylabel('Count')
for i, v in enumerate([sum(y_train_smote == 0), sum(y_train_smote == 1)]):
    axes[1].text(i, v + 20, str(v), ha='center', fontweight='bold')

plt.tight_layout()
plt.show()

print(f'Before SMOTE: {len(y_train)} samples')
print(f'After SMOTE:  {len(y_train_smote)} samples')

### 3c. Train on SMOTE-balanced data and evaluate

In [ ]:
smote_model = LogisticRegression(max_iter=1000, random_state=42)
smote_model.fit(X_train_smote, y_train_smote)

smote_results = evaluate_model(smote_model, X_test, y_test, 'SMOTE Logistic Regression')

print('Comparison with Baseline:')
for metric in ['accuracy', 'precision', 'recall', 'f1', 'auc']:
    diff = smote_results[metric] - baseline_results[metric]
    print(f'  {metric:>10s}: {diff:+.4f}')

### 3d. Try different SMOTE ratios

In [ ]:
smote_ratios = [0.1, 0.2, 0.5, 1.0]
smote_ratio_results = []

for ratio in smote_ratios:
    smote_r = SMOTE(sampling_strategy=ratio, random_state=42)
    X_res, y_res = smote_r.fit_resample(X_train, y_train)
    
    model_r = LogisticRegression(max_iter=1000, random_state=42)
    model_r.fit(X_res, y_res)
    
    res = evaluate_model(model_r, X_test, y_test, f'SMOTE ratio={ratio}')
    res['ratio'] = ratio
    smote_ratio_results.append(res)

In [ ]:
# Compare metrics across SMOTE ratios
ratios = [r['ratio'] for r in smote_ratio_results]
fig, ax = plt.subplots(figsize=(8, 5))

for metric in ['precision', 'recall', 'f1', 'auc']:
    vals = [r[metric] for r in smote_ratio_results]
    ax.plot(ratios, vals, 'o-', label=metric.capitalize())

ax.set_xlabel('SMOTE Sampling Strategy (ratio)')
ax.set_ylabel('Score')
ax.set_title('Effect of SMOTE Ratio on Model Performance')
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

### 3e. Discussion

Increasing the SMOTE ratio generates more synthetic minority samples, which generally improves recall (the model becomes better at detecting strokes) but decreases precision (more false positives). The F1 score captures the trade-off between the two. AUC tends to be relatively stable across ratios since it evaluates the ranking quality of predicted probabilities. A moderate ratio (e.g., 0.2-0.5) often provides the best balance between recall and precision for clinical applications.

## 4. Cost-Sensitive Learning

### 4a/b/c. Balanced class weights

In [ ]:
# Train with class_weight='balanced'
balanced_model = LogisticRegression(max_iter=1000, random_state=42, class_weight='balanced')
balanced_model.fit(X_train, y_train)

balanced_results = evaluate_model(balanced_model, X_test, y_test, 'Balanced Class Weight')

print('Comparison with Baseline:')
for metric in ['accuracy', 'precision', 'recall', 'f1', 'auc']:
    diff = balanced_results[metric] - baseline_results[metric]
    print(f'  {metric:>10s}: {diff:+.4f}')

print()
print('Comparison with SMOTE (ratio=1.0):')
for metric in ['accuracy', 'precision', 'recall', 'f1', 'auc']:
    diff = balanced_results[metric] - smote_results[metric]
    print(f'  {metric:>10s}: {diff:+.4f}')

### 4d. Custom class weights

In [ ]:
custom_weights = [1, 2, 5, 10, 20]
custom_results = []

for w in custom_weights:
    cw_model = LogisticRegression(max_iter=1000, random_state=42, class_weight={0: 1, 1: w})
    cw_model.fit(X_train, y_train)
    
    res = evaluate_model(cw_model, X_test, y_test, f'class_weight={{0:1, 1:{w}}}')
    res['weight'] = w
    custom_results.append(res)

In [ ]:
# Plot how metrics change with weight
weights = [r['weight'] for r in custom_results]

fig, ax = plt.subplots(figsize=(8, 5))
for metric in ['precision', 'recall', 'f1']:
    vals = [r[metric] for r in custom_results]
    ax.plot(weights, vals, 'o-', label=metric.capitalize(), linewidth=2)

ax.set_xlabel('Weight on Minority Class (stroke=1)')
ax.set_ylabel('Score')
ax.set_title('Effect of Class Weight on Precision, Recall, and F1')
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

### 4e. Discussion

Increasing the weight on the minority class (stroke=1) makes misclassifying stroke cases more costly, which pushes the model to predict more positives. This increases **recall** (fewer missed strokes) at the expense of **precision** (more false alarms). The F1 score initially rises as recall improves, then drops once precision degrades too much. The optimal weight depends on the application: in medical settings where missing a stroke is dangerous, higher recall is preferred even at the cost of more false positives.

## 5. Report

### 5a. Summary of all methods

In [ ]:
# Collect all key results
all_results = [baseline_results, smote_results, balanced_results]
# Add best SMOTE ratio and best custom weight
best_smote = max(smote_ratio_results, key=lambda r: r['f1'])
best_custom = max(custom_results, key=lambda r: r['f1'])
all_results.extend([best_smote, best_custom])

summary_df = pd.DataFrame([
    {k: v for k, v in r.items() if k in ['name', 'accuracy', 'precision', 'recall', 'f1', 'auc']}
    for r in all_results
]).set_index('name')

print('=== Summary of All Methods ===')
print(summary_df.round(4).to_string())

In [ ]:
# Combined ROC and PR curves
plot_curves(all_results, 'All Methods: ')

### 5b/c. Discussion

**Which technique(s) performed best?**

- The **baseline** model achieves high accuracy but very low recall, meaning it misses most stroke cases. This is typical in imbalanced datasets where the model learns to predict the majority class.
- **SMOTE** significantly improves recall by oversampling the minority class, at the cost of some precision. Moderate sampling ratios (0.2-0.5) often yield the best F1 trade-off.
- **Cost-sensitive learning** (`class_weight='balanced'`) achieves similar improvements to SMOTE without needing to generate synthetic data. Custom weights provide fine-grained control.
- Both SMOTE and cost-sensitive learning improve AUC and recall compared to the baseline, while the choice between them depends on the specific application requirements.

**Implications of class imbalance in medical prediction:**

In stroke prediction, a **false negative** (missing a real stroke case) is far more dangerous than a **false positive** (flagging a healthy patient for further testing). Therefore:
- **Recall** is the most important metric: we want to catch as many stroke cases as possible.
- **Precision** matters to avoid overwhelming clinicians with false alarms, but it is secondary.
- **Accuracy** is misleading with <5% prevalence and should not be the primary metric.
- The **precision-recall curve** and **F1 score** are more informative than ROC/AUC in this setting.

For clinical deployment, one should tune the decision threshold (or class weights) to maximize recall subject to an acceptable false positive rate, potentially guided by clinical cost-benefit analysis.